In [1]:
import os, io, ssl, json, re, sqlite3, zipfile, urllib.request, textwrap, operator
from typing import Annotated, Literal
from typing_extensions import TypedDict
import pandas as pd
from dotenv import load_dotenv

import truststore
truststore.inject_into_ssl()

# Prints the wall-clock time under every cell.
%load_ext autotime


def pretty_print(*args, width=95):
    """Reflow long prose to `width`, but leave tables / SQL output untouched."""
    text = " ".join(str(a) for a in args)
    if "\n" in text.strip("\n") or re.search(r"\S  +\S", text):
        print(text)
    else:
        print(textwrap.fill(text.strip(), width=width))


load_dotenv("/Users/shivam13juna/Documents/scaler/GEN_AI_REF/openai_key.env")
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found — check the openai_key.env path."

WORKER_MODEL = "gpt-4.1-nano"      # the cheap workhorse used for every agent below
REVIEWER_MODEL = "gpt-4.1-mini"    # the independent judge in P6
pretty_print(f"API key loaded.  worker={WORKER_MODEL}  reviewer={REVIEWER_MODEL}")

API key loaded.  worker=gpt-4.1-nano  reviewer=gpt-4.1-mini
time: 972 µs (started: 2026-09-12 21:10:08 +05:30)


In [3]:
DB_PATH = "online_retail.db"

# The three tools, as ordinary Python. Every framework in this notebook wraps THESE.
def list_tables():
    """Return the names of every table in the database."""
    connection = sqlite3.connect(DB_PATH)
    names = [row[0] for row in connection.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
    connection.close()
    return ", ".join(names)


def get_schema(table):
    """Return one table's columns (name + type)."""
    connection = sqlite3.connect(DB_PATH)
    columns = connection.execute(f"PRAGMA table_info({table})").fetchall()
    connection.close()
    if not columns:
        return f"No such table: {table}"
    return f"Table '{table}': " + ", ".join(f"{col[1]} ({col[2]})" for col in columns)


def run_sql(query, max_rows=20):
    """Run a read-only query and return rows as text — or the error message as text."""
    # mode=ro opens the file READ-ONLY: SQLite itself refuses any write, whoever asked for it.
    # This is the one tool that runs SQL the model wrote, so it is the one that needs the lock.
    connection = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
    try:
        cursor = connection.execute(query)
        if cursor.description is None:
            return "OK (no rows returned)."
        column_names = [description[0] for description in cursor.description]
        rows = cursor.fetchmany(max_rows)
        body = "\n".join(" | ".join(str(value) for value in row) for row in rows) or "(0 rows)"
        return f"{' | '.join(column_names)}\n{body}"
    except Exception as error:
        # Errors come back as TEXT, not exceptions, so an agent can read and recover from them.
        return f"SQL ERROR: {type(error).__name__}: {error}"
    finally:
        connection.close()


print(list_tables())
print(get_schema("invoices"))
print(run_sql("SELECT COUNT(*) AS cancelled FROM invoices WHERE is_cancelled = 1"))

invoices, line_items, products
Table 'invoices': invoice_no (TEXT), customer_id (REAL), invoice_ts (TEXT), country (TEXT), is_cancelled (INTEGER)
cancelled
3836
time: 2.91 ms (started: 2026-09-12 21:11:05 +05:30)


# LangChain

In [ ]:
#{"role": "system", "content": f"You are a helpful assistant that answers questions about the database at {DB_PATH}.  You have access to three tools: list_tables(), get_schema(table), and run_sql(query).  You can only use these tools to answer questions.  You cannot access the database directly.",
#"role": "user", "content": "What tables are in the database?"
#"role": "assistant", "content": list_tables()}

In [ ]:
from openai import OpenAI

client = OpenAI()

conversation = [
    {
        "role": "system",
        "content": "You are a terse data analyst.",
    },
    {
        "role": "user",
        "content": "Name the three tables in a retail schema, in one line.",
    },
]

response = client.responses.create(
    model=WORKER_MODEL,
    input=conversation,
    temperature=0,
)

pretty_print("reply:", response.output_text)

print("\nreply type:", type(response).__name__)

In [4]:
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

# "openai:gpt-4.1-nano" — the provider prefix is the whole abstraction. Point this at
# "anthropic:claude-..." and nothing else in the notebook needs to change.
language_model = init_chat_model(f"openai:{WORKER_MODEL}", temperature=0)

# Messages are typed objects rather than {"role": ..., "content": ...} dictionaries.
conversation = [
    SystemMessage("You are a terse data analyst."),
    HumanMessage("Name the three tables in a retail schema, in one line."),
]
model_reply = language_model.invoke(conversation)
pretty_print("reply:", model_reply.content)
print("\nreply type:", type(model_reply).__name__)

reply: Customers, Orders, Products

reply type: AIMessage
time: 5.69 s (started: 2026-09-12 21:13:23 +05:30)


In [5]:
from langchain.tools import tool


# Each wrapper delegates to the plain function from P0 — no logic is duplicated.
@tool
def sql_list_tables() -> str:
    """List all tables in the retail database."""
    return list_tables()


@tool
def sql_get_schema(table: str) -> str:
    """Show the columns and their types for one table."""
    return get_schema(table)


@tool
def sql_run(query: str) -> str:
    """Run a read-only SQLite query and return the rows, or a 'SQL ERROR: ...' string."""
    return run_sql(query)


DATABASE_TOOLS = [sql_list_tables, sql_get_schema, sql_run]

# This is the generated schema — exactly what we hand-wrote as JSON before.
print("tool name       :", sql_run.name)
print("description     :", sql_run.description)
print("generated schema:", sql_run.args_schema.model_json_schema()["properties"])

tool name       : sql_run
description     : Run a read-only SQLite query and return the rows, or a 'SQL ERROR: ...' string.
generated schema: {'query': {'title': 'Query', 'type': 'string'}}
time: 43.7 ms (started: 2026-09-12 21:18:00 +05:30)


In [6]:
# Binding attaches the schemas to the model. The model still cannot RUN anything —
# it can only reply with a request, exactly as in the raw OpenAI protocol.
model_with_tools = language_model.bind_tools(DATABASE_TOOLS)
tool_request = model_with_tools.invoke("How many invoices are cancelled?")

print("content    :", repr(tool_request.content))       # usually empty — it wants a tool first
print("tool_calls :", tool_request.tool_calls)

content    : ''
tool_calls : [{'name': 'sql_list_tables', 'args': {}, 'id': 'call_0qXjg3HFF4RRp8QL6nmRjGBJ', 'type': 'tool_call'}]
time: 1.4 s (started: 2026-09-12 21:19:34 +05:30)


In [8]:
from langchain.agents import create_agent
from IPython.display import Image, display

# The question this notebook asks of every agent it builds.
BUSINESS_QUESTION = "What was our total revenue, excluding cancelled orders?"

# One instruction string, shared by all three frameworks, so the comparison in P9 is fair.
# The tool ORDER is spelled out deliberately: left to itself the model will guess a plausible
# table name, query it, and report that the data is missing. Naming the discovery steps costs
# one sentence and removes that whole failure mode.
AGENT_INSTRUCTIONS = (
    "You are InsightAgent, a data analyst for an online-retail store. "
    "Always work in this order: first list the tables, then inspect the schema of every table "
    "you intend to use, and only then write SQL. Never guess a table or column name. "
    "Revenue must EXCLUDE cancelled invoices (invoices.is_cancelled = 1). "
    "State the final answer clearly, including the number."
)

prebuilt_agent = create_agent(
    model=f"openai:{REVIEWER_MODEL}",
    tools=DATABASE_TOOLS,
    system_prompt=AGENT_INSTRUCTIONS,
)


# The input and output are both a message list under the key "messages".
agent_result = prebuilt_agent.invoke({"messages": [HumanMessage(BUSINESS_QUESTION)]})
pretty_print("ANSWER:", agent_result["messages"][-1].content)

ANSWER: The total revenue, excluding cancelled orders, is 10,644,560.42.
time: 5.15 s (started: 2026-09-12 21:21:32 +05:30)


In [9]:
# Every intermediate step is in the returned message list — nothing is hidden.
# This is the same trace we printed by hand from a manual loop.
for message in agent_result["messages"]:
    label = type(message).__name__
    if isinstance(message, AIMessage) and message.tool_calls:
        print(f"  {label:14s} → calls {[c['name'] for c in message.tool_calls]}")
    elif isinstance(message, ToolMessage):
        print(f"  {label:14s} ← {message.content[:70]!r}")
    else:
        print(f"  {label:14s}   {str(message.content)[:70]!r}")

  HumanMessage     'What was our total revenue, excluding cancelled orders?'
  AIMessage      → calls ['sql_list_tables']
  ToolMessage    ← 'invoices, line_items, products'
  AIMessage      → calls ['sql_get_schema', 'sql_get_schema']
  ToolMessage    ← "Table 'invoices': invoice_no (TEXT), customer_id (REAL), invoice_ts (T"
  ToolMessage    ← "Table 'line_items': invoice_no (TEXT), stock_code (TEXT), quantity (IN"
  AIMessage      → calls ['sql_run']
  ToolMessage    ← 'total_revenue\n10644560.424'
  AIMessage        'The total revenue, excluding cancelled orders, is 10,644,560.42.'
time: 552 µs (started: 2026-09-12 21:21:48 +05:30)
